# 1. API Key 로드

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

# 2. Few-Shot examples

https://python.langchain.com/docs/how_to/few_shot_examples/

- Few-Shotting: 
    - 몇 개의 예시를 LLM에 제시하는 것 
    - 서비스 개발 시 LLM 모델에 전달될 일관된 프롬프트와 LLM모델의 답변형식이 정해져있을 경우 해당 형식을 LLM에 학습시키는 프롬프팅 방식 
- 간단하지만 강력한 기법으로 경우에 따라 모델의 성능을 크게 향상시킬 수 있음 
- FewShotChatMessagePromptTemplate 객체를 통해 Few-Shot prompt 생성 
    - 구성: examples와 example_prompt를 전달하여 FewShotChatMessagePromptTemplate통해 few-shot prompt를 완성시켜줌
        - examples: 미리 정의된 User 프롬프트 및 AI 답변 예제(examples) 텍스트
        - example_prompt: examples가 작성될 Role베이스의 채팅 템플릿

## 2.1 Zero-Shot: Few-Shot 없이 답변 생성해보기 

In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-nano", temperature=0)

response = model.invoke("2 🦜 9는 뭐야?")
print(response.content)

의도치가 애매하네요. 🦜가 어떤 의미로 쓰인 건지에 따라 다르게 해석될 수 있어요. 몇 가지 가능성을 적어둘게요. 맥락을 알려주시면 바로 맞춰드릴게요.

- 2 to the 9th (2의 9제곱): 2^9 = 512
- 2 × 9: 18
- 2부터 9까지의 합(포함): 44
- 2부터 9까지의 수 개수: 8개

어떤 맥락에서 쓰신 건가요? 수학 문제인지, 암호 같은 퍼즐인지 알려주시면 정확히 답해드리겠습니다.


## 2.2 Few-Shot 활용하여 답변 생성해보기

### 2.2.1 예제1

1. few-shot 생성

In [4]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 1. Few-Shot에 쓰일 examples리스트 생성
# - examples list는 user prompt와 ai prompt에 채워넣을 변수로 구성된 딕셔너리가 요소로 들어감

examples = [
    {"input_data": "2 🦜 9는 뭐야?", "ai": "2 🦜 9는 18입니다."},
    {"input_data": "4 🦜 8는 뭐야?", "ai": "4 🦜 8는 32입니다."},
    {"input_data": "5 🦜 5는 뭐야?", "ai": "5 🦜 5는 25입니다."}
]

# 2. examples를 전달할 template 생성
example_template = ChatPromptTemplate([
    ("user", "{input_data}"),
    ("ai", "{ai}")
])

# 3. Few-Shot 프롬프트 생성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template
)

In [5]:
# 최종 prompt에 전달될 fewshot_prompt 확인 
print(fewshot_prompt.invoke({}).to_messages())

[HumanMessage(content='2 🦜 9는 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='2 🦜 9는 18입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='4 🦜 8는 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='4 🦜 8는 32입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='5 🦜 5는 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='5 🦜 5는 25입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [6]:
# 위에 출력된 내용과 같은 예제 내용. 
"""
[
    HumanMessage(content='2 🦜 3는 뭐야?', additional_kwargs={}, response_metadata={}), 
    AIMessage(content='2 🦜 3는 6입니다.', additional_kwargs={}, response_metadata={}), 
    HumanMessage(content='4 🦜 8는 뭐야?', additional_kwargs={}, response_metadata={}), 
    AIMessage(content='4 🦜 8는 32입니다.', additional_kwargs={}, response_metadata={}), 
    HumanMessage(content='5 🦜 5는 뭐야?', additional_kwargs={}, response_metadata={}), 
    AIMessage(content='5 🦜 5는 25입니다.', additional_kwargs={}, response_metadata={})
]
"""

"\n[\n    HumanMessage(content='2 🦜 3는 뭐야?', additional_kwargs={}, response_metadata={}), \n    AIMessage(content='2 🦜 3는 6입니다.', additional_kwargs={}, response_metadata={}), \n    HumanMessage(content='4 🦜 8는 뭐야?', additional_kwargs={}, response_metadata={}), \n    AIMessage(content='4 🦜 8는 32입니다.', additional_kwargs={}, response_metadata={}), \n    HumanMessage(content='5 🦜 5는 뭐야?', additional_kwargs={}, response_metadata={}), \n    AIMessage(content='5 🦜 5는 25입니다.', additional_kwargs={}, response_metadata={})\n]\n"

2. model에 few-shot을 전달하여 원하는 형식으로 답변 받기 

In [7]:
from langchain_core.output_parsers import StrOutputParser

# 1. 최종 template 생성
final_template = ChatPromptTemplate([
    ("system", "당신은 놀라운 수학 마법사입니다."),
    fewshot_prompt,
    ("user", "{input_data}") # input_data: 퓨샷 포맷과 동일하게 작성해야 지피티가 퓨샷의 예제를 참고해서 답변을해줌. 포맷이 다르면 지피티도 전혀 다른 답변을 함.
])

# 2. model 정의
model = ChatOpenAI(model="gpt-5-nano")

# 3. chain 생성
chain = final_template | model | StrOutputParser()

# 4. 응답 생성
response = chain.invoke("7 🦜 6은 뭐야?")
print(response)

7 🦜 6은 42입니다. (두 수를 곱한 값)


### 2.2.2 예제2: 금융 

- 거래 내역 자동 분류 및 분석 
- 활용 시나리오: 고객의 카드 사용 내역을 자동으로 카테고리 분류하고 소비패턴 분석 

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [9]:
# examples
examples = [
    {
        "input_data": """
2024-01-10: 스타벅스 6,500원
2024-01-11: 쿠팡 58,000원
2024-01-12: GS칼텍스 65,000원
        """,
        "output": """ 
[거래 내역 분석 리포트]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 카테고리별 지출
- 식음료: 65,000원 (5.0%)
- 쇼핑: 58,000원 (44.6%)
- 교통/주유: 65,000원 (50.4%)

💸총 지출: 129,500원
📈최다 지출 카테고리: 교통/주유: (65,000원)

💡절약 TIP
- 주유 할인 카드 활용 시 약 3,250원 절감 가능
- 카테고리: 교통/주유 > 쇼핑 > 식음료순
        """
    },
    {
        "input_data": """
2024-02-01: 넷플릭스 13,500원
2024-02-03: 올리브영 42,000원
2024-02-05: 배달의민족 28,000원
2024-02-07: 카카오T 15,500원
        """,
        "output": """
[ 거래 내역 분석 리포트 ]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 카테고리별 지출
- 구독서비스: 13,500원 (13.6%)
- 뷰티/건강: 42,000원 (42.4%)
- 배달음식: 28,000원 (28.3%)
- 교통: 15,500원 (15.7%)

💰 총 지출: 99,000원
📈 최다 지출 카테고리: 뷰티/건강 (42,000원)

💡 절약 TIP
- 배달음식 주 1회 줄이면 월 28,000원 절감
- 카테고리: 뷰티/건강 > 배달음식 > 교통 > 구독 순
        """
    },
    {
        "input_data": """
2024-03-10: 신세계백화점 185,000원
2024-03-12: 스타벅스 7,500원
2024-03-15: 이마트 68,000원
        """,
        "output": """
[ 거래 내역 분석 리포트 ]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 카테고리별 지출
- 백화점/쇼핑: 185,000원 (71.1%)
- 식음료: 7,500원 (2.9%)
- 생필품: 68,000원 (26.1%)

💰 총 지출: 260,500원
📈 최다 지출 카테고리: 백화점/쇼핑 (185,000원)

💡 절약 TIP
- 백화점 대신 아울렛 이용 시 약 37,000원 절감 가능
- 카테고리: 백화점/쇼핑 > 생필품 > 식음료 순
        """
    }
]

# example_template 작성
example_template = ChatPromptTemplate([
    ("user", "{input_data}"),
    ("ai", "{output}")
])

# Few-Shot 프롬프트 구성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template
)


In [10]:
# 최종 template 작성
system_prompt = """ 
당신은 금융 데이터 분석 AI입니다.
고객의 거래 내역을 받아 카테고리별로 분류하고, 소비 패턴을 분석하여 정해진 포맷으로 리포트를 작성하세요.

분석 원칙:
1. 거래처명을 보고 카테고리 자동 분류
2. 카테고리별 금액과 비율 계산
3. 절약 팁은 실용적이고 구체적으로
4. 포맷은 예시와 동일하게 유지
"""
final_template = ChatPromptTemplate([
    ("system", system_prompt),
    fewshot_prompt,
    ("user", "{input_data}")
])

# model 정의
model = ChatOpenAI(model="gpt-5-nano")

# chain 생성
chain = final_template | model | StrOutputParser()

# 응답 생성
input_data = """
2024-04-05: 쿠팡 로켓배송 78,000원
2024-04-07: 카페베네 8,500원
2024-04-10: 유니클로 89,000원
2024-04-12: 교보문고 35,000원
"""
response = chain.invoke(input_data)
print(response)


[ 거래 내역 분석 리포트 ]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 카테고리별 지출
- 쇼핑: 78,000원 (37.0%)
- 식음료: 8,500원 (4.0%)
- 패션/의류: 89,000원 (42.3%)
- 도서/문화: 35,000원 (16.6%)

💰 총 지출: 210,500원
📈 최다 지출 카테고리: 패션/의류 (89,000원)

💡 절약 TIP
- 쇼핑: 필요 물건 목록 작성 후 가격 비교와 무료배송/쿠폰 활용으로 비용 절감
- 식음료: 카페 방문 횟수를 주간 단위로 조정하고 집에서 커피를 만들면 큰 폭으로 줄일 수 있음
- 패션/의류: 정기 세일 기간 노려 구입, 중고나 아울렛 활용으로 동일 품질의 아이템을 더 저렴하게 확보
- 도서/문화: 도서관 대출이나 전자책 이용, 중고 서점 비교 구매로 비용 절감


# 3. 실습 

회의 시간에 메모한 회의 기록을 특정 포맷에 맞게 정리하여 출력해주도록 LLM을 학습시킨 후 아래의 주어진 회의 기록 예제(input_example)로 LLM의 응답을 받아보자

input_example = """\
"240415 오후 1시, JKL 회사의 재무 전략 회의 온라인으로 진행
참석자 재무팀장 박시은, 회계 담당자 오세훈, 외부 컨설턴트 윤지아
2024년 2분기 예산 조정안 검토 및 비용 절감 방안 논의를 중심으로 이뤄짐
박시은: 현재까지의 지출 현황을 보고
오세훈: 부서별 예산 사용 내역 상세히 설명
윤지아: 비용 절감 위한 프로세스 개선 아이디어 제시.
다음 조치에 대한 합의로 마무리됨"
"""

In [11]:
examples = [
    {
        "input": "231225, XYZ 회사의 마케팅 전략 회의 오후 3시 진행. 마케팅 팀장 김수진, 디지털 마케팅 담당자 박지민, 소셜 미디어 관리자 이준호 참석. 회의 목적: 2024년 상반기 마케팅 전략 수립, 새로운 소셜 미디어 캠페인에 대한 아이디어 논의. 김수진 - 최근 시장 동향에 대한 간략한 개요 제공, 각 팀원 - 자신의 분야에서의 전략적 아이디어 발표.",
        "ai": """
📋 회의록: XYZ 회사 마케팅 전략 회의
📅 일시: 2023년 12월 25일
🏢 장소: XYZ 회사 회의실
👥 참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
""",
    },
    {
    "input": "2024년 1월 10일, ABC 스타트업의 제품 개발 회의 오전 10시에 진행. 참석자: 제품 매니저 김현우, UX 디자이너 이서윤, 백엔드 개발자 정우성. 안건: 신규 앱 기능 추가 및 UI 개선 방향. \n김현우 - 지난 분기 사용자 피드백 공유 -> 개선이 필요한 부분을 언급\n이서윤 - 새로운 UI 시안 2가지를 제안.\n정우성 - 기술 구현 가능성, 예상 개발 일정 설명.",
    "ai": """
📋 회의록: ABC 스타트업 제품 개발 회의  
📅 일시: 2024년 1월 10일  
🏢 장소: ABC 사무실 회의실  
👥 참석자: 김현우 (제품 매니저), 이서윤 (UX 디자이너), 정우성 (백엔드 개발자)  

1. 개회  
   - 회의는 김현우 제품 매니저의 인사말로 시작.  
   - 회의 목적은 신규 기능 추가 및 UI 개선 논의.  

2. 사용자 피드백 공유 (김현우)  
   - 사용자 불편 사항 및 개선 요구사항 소개.  
   - 특히 로그인 UX와 탐색 속도에 대한 불만이 많았음.  

3. UI 개선안 발표 (이서윤)  
   - 새로운 UI 시안 2가지 제시.  
   - 간결한 네비게이션 구조와 접근성 향상에 중점.  

4. 기술 검토 (정우성)  
   - 각 UI 시안에 대한 구현 가능성 분석.  
   - 개발 기간은 약 3주 소요 예정.  

5. 결론  
   - 시안 1안으로 최종 결정.  
   - 개발 착수일은 1월 15일, 다음 리뷰 회의는 1월 31일로 확정.
"""
    },
    {
        "input": "2024년 2월 5일 오후 2시, DEF 기업의 인사팀이 신규 개발자 채용을 위한 회의 진행함.\n참석자: 인사팀장 조민지, 채용 담당자 배지훈, 기술 평가 담당자 최영준. \n조민지: 채용 공고에 대한 수정사항 공유\n배지훈: 현재 지원자 현황 보고.\n최영준: 기술 면접 항목, 기준 제안\n서류 통과 기준, 면접 일정에 대해 최종 논의 진행",
        "ai": """
📋 회의록: DEF 기업 인사팀 채용 회의  
📅 일시: 2024년 2월 5일  
🏢 장소: 인사팀 회의실  
👥 참석자: 조민지 (인사팀장), 배지훈 (채용 담당자), 최영준 (기술 평가 담당자)  

1. 개회  
- 회의 목적은 신규 개발자 채용 절차 논의.  

2. 채용 공고 수정안 (조민지)  
- 직무 기술서 및 우대사항 수정 내용 공유.  

3. 지원자 현황 보고 (배지훈)  
- 현재까지 총 48명 지원.  
- 서류 평가 진행 중이며, 1차 합격자 발표 예정일 공유.  

4. 기술 평가 기준 (최영준)  
- 코딩 테스트 항목과 면접 시 질문 예시 제안.  
- 실무 능력 평가 중심의 구조 설계.  

5. 결론  
- 서류 통과 기준 확정: 경력 2년 이상, 포트폴리오 필수.  
- 1차 면접 일정: 2월 13~15일  
- 면접 질문은 다음 주 공유 예정.
"""
    },
    {
    "input": "240302 오후 4시\nGHI 서비스팀 - 최근 고객 만족도 조사 결과 바탕으로 내부 회의 진행.\n고객 지원팀장 안세진, CS 담당자 김태리, 데이터 분석가 유민호 참석.\n유민호 - 최근 1분기 설문 결과를 발표\n김태리 - 자주 접수되는 불만 유형을 정리해서 공유\n안세진 - 해결 방안으로 고객 응대 매뉴얼 개정, 추가 교육 제안",
    "ai": """
📋 회의록: GHI 서비스팀 고객 만족 회의  
📅 일시: 2024년 3월 2일  
🏢 장소: GHI 본사 2층 회의실  
👥 참석자: 안세진 (고객 지원팀장), 김태리 (CS 담당자), 유민호 (데이터 분석가)  

1. 회의 개요  
   - 1분기 고객 만족도 조사 결과를 기반으로 서비스 품질 개선 논의.  

2. 설문 결과 발표 (유민호)  
   - 전체 만족도 78%  
   - 주요 불만: 응답 지연, 해결 속도  

3. 불만 유형 분석 (김태리)  
   - 주요 유형: 배송 지연, 불친절 응대, 처리 지연  
   - VOC 사례 일부 소개  

4. 개선안 제안 (안세진)  
   - 응대 매뉴얼 업데이트  
   - 전 직원 대상 고객 응대 재교육 실시 제안  

5. 종합 논의 및 결론  
   - 다음 분기 목표: 응답 시간 평균 20% 단축  
   - 3월 말까지 교육 완료, 4월 초부터 새 매뉴얼 적용  
   - 재조사 일정은 6월 초로 설정
"""
    }
]

In [12]:
# example_template 작성
example_template = ChatPromptTemplate([
    ("user", "{input}"),
    ("ai", "{ai}")
])

# Few-Shot 프롬프트 구성
fewshot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template
)


In [13]:
# 최종 template 작성
system_prompt = """ 
당신은 정확성을 추구하는 회의 기록 전문가입니다.
고객이 회의 시간에 메모한 회의 기록을 입력받아 특정 포맷에 맞게 정리하여 출력하세요.

분석 원칙:
1. 상단에는 회의록, 일시, 장소, 참석자를 작성하세요.
2. 상단에 작성된 회의록은 회의의 가장 큰 topic으로 1줄로 간략하게 작성하세요.
3. 회의 내용을 요약할때는 반드시 해당 내용 발표자의 이름을 입력하세요.
4. 각 카테고리의 요약 내용은 3줄 이하로 간략히 작성하세요.
5. 결론은 구체적인 날짜, 수치를 반드시 입력하세요.

출력 예시:

📋 회의록:
📅 일시: 
🏢 장소:
👥 참석자:

1. 회의 개요  
2. 설문 결과 발표 (발표자) 
3. 불만 유형 분석  
4. 개선안 제안 
5. 종합 논의 및 결론  
"""
final_template = ChatPromptTemplate([
    ("system", system_prompt),
    fewshot_prompt,
    ("user", "{input}")
])

# model 정의
model = ChatOpenAI(model="gpt-5-nano")

# chain 생성
chain = final_template | model | StrOutputParser()

# 응답 생성
input_example = """\
"240415 오후 1시, JKL 회사의 재무 전략 회의 온라인으로 진행
참석자 재무팀장 박시은, 회계 담당자 오세훈, 외부 컨설턴트 윤지아
2024년 2분기 예산 조정안 검토 및 비용 절감 방안 논의를 중심으로 이뤄짐
박시은: 현재까지의 지출 현황을 보고
오세훈: 부서별 예산 사용 내역 상세히 설명
윤지아: 비용 절감 위한 프로세스 개선 아이디어 제시.
다음 조치에 대한 합의로 마무리됨"
"""
response = chain.invoke(input_example)
print(response)

📋 회의록: JKL 회사 2024년 2분기 예산 조정 및 비용 절감 논의  
📅 일시: 2024년 4월 15일 오후 1시  
🏢 장소: 온라인 회의  
👥 참석자: 박시은 (재무팀장), 오세훈 (회계 담당자), 윤지아 (외부 컨설턴트)

1. 개회 (발표자: 박시은)  
- 회의 목적은 2024년 2분기 예산 조정안 검토 및 비용 절감 논의.

2. 예산 조정안 검토 (발표자: 오세훈)  
- 부서별 예산 사용 내역 상세 설명.  
- 조정 필요 지출 항목 식별 및 재배치 제안.  
- 2분기 목표 절감액에 대한 의견 수렴.

3. 비용 절감 아이디어 제시 (발표자: 윤지아)  
- 프로세스 개선 및 자동화 도입 아이디어 제시.  
- 공급업체 재협상 및 불필요 지출 최소화 제안.  
- 비용 구조 재구성에 따른 단기/중기 효과 예시.

4. 결론 및 차기 조치 (발표자: 박시은)  
- 차기 조치: 2024년 4월 22일에 예산 조정안 최종 확정.  
- 비용 절감 목표: 8% 달성.  
- 실행 계획 및 반영 일정: 4월 29일까지 세부 실행 계획 수립 및 시스템 반영.


In [14]:
# 선생님 답변 

examples = [
    {
        "input": "231225, XYZ 회사의 마케팅 전략 회의 오후 3시 진행. 마케팅 팀장 김수진, 디지털 마케팅 담당자 박지민, 소셜 미디어 관리자 이준호 참석. 회의 목적: 2024년 상반기 마케팅 전략 수립, 새로운 소셜 미디어 캠페인에 대한 아이디어 논의. 김수진 - 최근 시장 동향에 대한 간략한 개요 제공, 각 팀원 - 자신의 분야에서의 전략적 아이디어 발표.",
        "ai": """
📋 회의록: XYZ 회사 마케팅 전략 회의
📅 일시: 2023년 12월 25일
🏢 장소: XYZ 회사 회의실
👥 참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의는 김수진 팀장의 개회사로 시작됨.
   - 회의의 목적은 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의.

2. 시장 동향 개요 (김수진)
   - 김수진 팀장은 최근 시장 동향에 대한 분석을 제시.
   - 소비자 행동 변화와 경쟁사 전략에 대한 통찰 공유.

3. 디지털 마케팅 전략 (박지민)
   - 박지민은 디지털 마케팅 전략에 대해 발표.
   - 온라인 광고와 SEO 최적화 방안에 중점을 둠.

4. 소셜 미디어 캠페인 (이준호)
   - 이준호는 새로운 소셜 미디어 캠페인에 대한 아이디어를 제안.
   - 인플루언서 마케팅과 콘텐츠 전략에 대한 계획을 설명함.

5. 종합 논의
   - 팀원들 간의 아이디어 공유 및 토론.
   - 각 전략에 대한 예산 및 자원 배분에 대해 논의.

6. 마무리
   - 다음 회의 날짜 및 시간 확정.
   - 회의록 정리 및 배포는 박지민 담당.
""",
    },
    {
    "input": "2024년 1월 10일, ABC 스타트업의 제품 개발 회의 오전 10시에 진행. 참석자: 제품 매니저 김현우, UX 디자이너 이서윤, 백엔드 개발자 정우성. 안건: 신규 앱 기능 추가 및 UI 개선 방향. \n김현우 - 지난 분기 사용자 피드백 공유 -> 개선이 필요한 부분을 언급\n이서윤 - 새로운 UI 시안 2가지를 제안.\n정우성 - 기술 구현 가능성, 예상 개발 일정 설명.",
    "ai": """
📋 회의록: ABC 스타트업 제품 개발 회의  
📅 일시: 2024년 1월 10일  
🏢 장소: ABC 사무실 회의실  
👥 참석자: 김현우 (제품 매니저), 이서윤 (UX 디자이너), 정우성 (백엔드 개발자)  

1. 개회  
   - 회의는 김현우 제품 매니저의 인사말로 시작.  
   - 회의 목적은 신규 기능 추가 및 UI 개선 논의.  

2. 사용자 피드백 공유 (김현우)  
   - 사용자 불편 사항 및 개선 요구사항 소개.  
   - 특히 로그인 UX와 탐색 속도에 대한 불만이 많았음.  

3. UI 개선안 발표 (이서윤)  
   - 새로운 UI 시안 2가지 제시.  
   - 간결한 네비게이션 구조와 접근성 향상에 중점.  

4. 기술 검토 (정우성)  
   - 각 UI 시안에 대한 구현 가능성 분석.  
   - 개발 기간은 약 3주 소요 예정.  

5. 결론  
   - 시안 1안으로 최종 결정.  
   - 개발 착수일은 1월 15일, 다음 리뷰 회의는 1월 31일로 확정.
"""
    },
    {
        "input": "2024년 2월 5일 오후 2시, DEF 기업의 인사팀이 신규 개발자 채용을 위한 회의 진행함.\n참석자: 인사팀장 조민지, 채용 담당자 배지훈, 기술 평가 담당자 최영준. \n조민지: 채용 공고에 대한 수정사항 공유\n배지훈: 현재 지원자 현황 보고.\n최영준: 기술 면접 항목, 기준 제안\n서류 통과 기준, 면접 일정에 대해 최종 논의 진행",
        "ai": """
📋 회의록: DEF 기업 인사팀 채용 회의  
📅 일시: 2024년 2월 5일  
🏢 장소: 인사팀 회의실  
👥 참석자: 조민지 (인사팀장), 배지훈 (채용 담당자), 최영준 (기술 평가 담당자)  

1. 개회  
- 회의 목적은 신규 개발자 채용 절차 논의.  

2. 채용 공고 수정안 (조민지)  
- 직무 기술서 및 우대사항 수정 내용 공유.  

3. 지원자 현황 보고 (배지훈)  
- 현재까지 총 48명 지원.  
- 서류 평가 진행 중이며, 1차 합격자 발표 예정일 공유.  

4. 기술 평가 기준 (최영준)  
- 코딩 테스트 항목과 면접 시 질문 예시 제안.  
- 실무 능력 평가 중심의 구조 설계.  

5. 결론  
- 서류 통과 기준 확정: 경력 2년 이상, 포트폴리오 필수.  
- 1차 면접 일정: 2월 13~15일  
- 면접 질문은 다음 주 공유 예정.
"""
    },
    {
    "input": "240302 오후 4시\nGHI 서비스팀 - 최근 고객 만족도 조사 결과 바탕으로 내부 회의 진행.\n고객 지원팀장 안세진, CS 담당자 김태리, 데이터 분석가 유민호 참석.\n유민호 - 최근 1분기 설문 결과를 발표\n김태리 - 자주 접수되는 불만 유형을 정리해서 공유\n안세진 - 해결 방안으로 고객 응대 매뉴얼 개정, 추가 교육 제안",
    "ai": """
📋 회의록: GHI 서비스팀 고객 만족 회의  
📅 일시: 2024년 3월 2일  
🏢 장소: GHI 본사 2층 회의실  
👥 참석자: 안세진 (고객 지원팀장), 김태리 (CS 담당자), 유민호 (데이터 분석가)  

1. 회의 개요  
   - 1분기 고객 만족도 조사 결과를 기반으로 서비스 품질 개선 논의.  

2. 설문 결과 발표 (유민호)  
   - 전체 만족도 78%  
   - 주요 불만: 응답 지연, 해결 속도  

3. 불만 유형 분석 (김태리)  
   - 주요 유형: 배송 지연, 불친절 응대, 처리 지연  
   - VOC 사례 일부 소개  

4. 개선안 제안 (안세진)  
   - 응대 매뉴얼 업데이트  
   - 전 직원 대상 고객 응대 재교육 실시 제안  

5. 종합 논의 및 결론  
   - 다음 분기 목표: 응답 시간 평균 20% 단축  
   - 3월 말까지 교육 완료, 4월 초부터 새 매뉴얼 적용  
   - 재조사 일정은 6월 초로 설정
"""
    }
]


example_template = ChatPromptTemplate([
    ("user", "{input}"),
    ("ai", "{ai}")
])

fewshot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_template
)


In [15]:
system_prompt = """\
당신은 회의록 작성 전문가입니다. 주어진 사용자의 입력을 바탕으로 회의록을 작성하되, 사용자 입력에 작성되지 않은 항목을 생략해주세요.\
"""

final_template = ChatPromptTemplate([
    ("system", system_prompt),
    fewshot_prompt,
    ("user", "{input}")
])

model = ChatOpenAI(model="gpt-5-nano")

chain = final_template | model | StrOutputParser()

input_example = """\
"240415 오후 1시, JKL 회사의 재무 전략 회의 온라인으로 진행
참석자 재무팀장 박시은, 회계 담당자 오세훈, 외부 컨설턴트 윤지아
2024년 2분기 예산 조정안 검토 및 비용 절감 방안 논의를 중심으로 이뤄짐
박시은: 현재까지의 지출 현황을 보고
오세훈: 부서별 예산 사용 내역 상세히 설명
윤지아: 비용 절감 위한 프로세스 개선 아이디어 제시.
다음 조치에 대한 합의로 마무리됨
"""

response = chain.invoke(input_example)
print(response)


📋 회의록: JKL 회사 재무 전략 회의  
📅 일시: 2024년 4월 15일  
🌐 방식: 온라인  
👥 참석자: 박시은 (재무팀장), 오세훈 (회계 담당자), 윤지아 (외부 컨설턴트)  

1. 안건  
- 2024년 2분기 예산 조정안 검토 및 비용 절감 방안 논의

2. 진행 내용  
- 박시은: 현재까지의 지출 현황 보고  
- 오세훈: 부서별 예산 사용 내역 상세히 설명  
- 윤지아: 비용 절감 위한 프로세스 개선 아이디어 제시

3. 결론  
- 다음 조치에 대한 합의로 마무리됨
